# A4 Faster R-CNN — core

Adapted from University of Michigan EECS 498/598, Deep Learning for Computer Vision (Winter 2022), by Justin Johnson and course staff.

**Static validation only; not executed.** Enable the run switches explicitly; generated files go to `core_results/`.

In [ ]:
from pathlib import Path
import os
import sys
import torch
import matplotlib.pyplot as plt
candidates = []
for base in (Path.cwd(), *Path.cwd().parents,
             Path('/content/drive/MyDrive/umich-dlcv-wi2022')):
    candidates.extend((base, base / 'assignments' / 'a4'))
ASSIGNMENT_DIR = next((p.resolve() for p in candidates if (p / 'two_stage_detector_core.py').is_file()), None)
if ASSIGNMENT_DIR is None:
    raise FileNotFoundError('Open from the project/assignment directory, or mount the canonical Colab Drive project first.')
loaded_helpers = sys.modules.get('eecs598')
if loaded_helpers is not None:
    loaded_path = Path(loaded_helpers.__file__).resolve().parent
    if loaded_path != (ASSIGNMENT_DIR / 'eecs598').resolve():
        raise RuntimeError('Restart the kernel before switching assignments: eecs598 is already loaded from another assignment.')
sys.path.insert(0, str(ASSIGNMENT_DIR))
os.chdir(ASSIGNMENT_DIR)
from eecs598.utils import reset_seed
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = ASSIGNMENT_DIR / 'core_results' / 'two_stage_detector'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_OVERFIT = False
ALLOW_DOWNLOAD = False
reset_seed(0)
plt.rcParams['image.cmap'] = 'gray'
print('Outputs:', OUTPUT_DIR)

## Data and model
VOC2007 non-difficult train/validation boxes, 224-pixel images, batch size 16. Use an existing dataset or explicitly allow its download.

In [ ]:
from a4_helper_core import VOC2007DetectionTiny, train_detector, inference_with_detector
from common_core import DetectorBackboneWithFPN
from one_stage_detector_core import FCOS
from two_stage_detector_core import RPN, FasterRCNN
NUM_CLASSES, BATCH_SIZE, IMAGE_SIZE = 20, 16, 224
NUM_WORKERS = 0  # Portable in Windows notebook kernels.

In [ ]:
def make_detector(channels=128):
    backbone = DetectorBackboneWithFPN(out_channels=channels)
    rpn = RPN(fpn_channels=channels, stem_channels=[channels, channels],
              batch_size_per_image=16, pre_nms_topk=500, post_nms_topk=200)
    return FasterRCNN(backbone, rpn, num_classes=NUM_CLASSES, roi_size=(7, 7),
                      stem_channels=[channels, channels], batch_size_per_image=32).to(DEVICE)

In [ ]:
if RUN_TRAINING or RUN_EVALUATION or RUN_OVERFIT:
    train_dataset = VOC2007DetectionTiny(str(ASSIGNMENT_DIR), 'train', download=ALLOW_DOWNLOAD, image_size=IMAGE_SIZE)
    val_dataset = VOC2007DetectionTiny(str(ASSIGNMENT_DIR), 'val', image_size=IMAGE_SIZE)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=1, num_workers=NUM_WORKERS)
    print('Training / validation images:', len(train_dataset), len(val_dataset))

## Optional overfit and full training

In [ ]:
if RUN_OVERFIT:
    reset_seed(0)
    indices = torch.linspace(0, len(train_dataset) - 1, steps=BATCH_SIZE * 10).long()
    small_loader = torch.utils.data.DataLoader(torch.utils.data.Subset(train_dataset, indices), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    overfit_detector = make_detector(64)
    train_detector(overfit_detector, small_loader, learning_rate=0.01, max_iters=1000, log_period=20, device=DEVICE)

In [ ]:
if RUN_TRAINING:
    reset_seed(0)
    detector = make_detector()
    train_detector(detector, train_loader, learning_rate=0.01, max_iters=9000, log_period=50, device=DEVICE)
    torch.save(detector.state_dict(), OUTPUT_DIR / 'rcnn_detector.pt')

## Validation and AP
Evaluation defaults to the core checkpoint. To inspect the ordinary checkpoint, explicitly set `EVAL_WEIGHTS = ASSIGNMENT_DIR / "rcnn_detector.pt"`; those results are not a core training run. Provide an existing Cartucho/mAP checkout at `MAP_SOURCE`; no repository is cloned or package installed.

In [ ]:
EVAL_WEIGHTS = OUTPUT_DIR / 'rcnn_detector.pt'
MAP_SOURCE = ASSIGNMENT_DIR / 'mAP'

In [ ]:
if RUN_EVALUATION:
    import shutil
    import subprocess
    if not EVAL_WEIGHTS.is_file():
        raise FileNotFoundError(f'Train first or explicitly choose an existing checkpoint: {EVAL_WEIGHTS}')
    if not (MAP_SOURCE / 'main.py').is_file():
        raise FileNotFoundError(f'An existing Cartucho/mAP evaluator is required at {MAP_SOURCE}; configure MAP_SOURCE before evaluation.')
    detector = make_detector()
    detector.load_state_dict(torch.load(EVAL_WEIGHTS, map_location='cpu', weights_only=True))
    small_indices = torch.linspace(0, len(val_dataset) - 1, steps=20).long()
    small_loader = torch.utils.data.DataLoader(torch.utils.data.Subset(val_dataset, small_indices), batch_size=1)
    inference_with_detector(detector, small_loader, val_dataset.idx_to_class, score_thresh=0.2, nms_thresh=0.5, device=DEVICE)
    evaluation_dir = OUTPUT_DIR / 'mAP'
    shutil.copytree(MAP_SOURCE, evaluation_dir, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('.git', '__pycache__', 'input', 'output', 'images'))
    inference_with_detector(detector, val_loader, val_dataset.idx_to_class,
                            score_thresh=0.2, nms_thresh=0.5, device=DEVICE,
                            output_dir=str(evaluation_dir / 'input'))
    subprocess.run([sys.executable, 'main.py', '--no-animation', '--no-plot'], cwd=evaluation_dir, check=True)
    print('Evaluation checkpoint:', EVAL_WEIGHTS)
    print('AP results:', evaluation_dir / 'output')